# Hybrid retrieval: Text & image embedding (ViDoRe V3 Industrial)

In [ ]:
!pip install -q datasets pillow pandas bm25s PyStemmer "sentence-transformers" "transformers"

## Load dataset

In [ ]:
from collections import defaultdict
from datasets import load_dataset

ds_corpus = load_dataset("vidore/vidore_v3_industrial", "corpus")
ds_labels = load_dataset("vidore/vidore_v3_industrial", "qrels")
ds_queries = load_dataset("vidore/vidore_v3_industrial", "queries")


def build_qrels(qrels) -> dict[int, set[int]]:
    out = defaultdict(set)
    for row in qrels:
        if row["score"] > 0:
            out[row["query_id"]].add(row["corpus_id"])
    return dict(out)


qrels_by_query = build_qrels(ds_labels["test"])  # query_id -> relevant corpus_ids (retrieval ground truth)
print(f"{len(qrels_by_query)} queries with relevance labels")

In [ ]:
import numpy as np

# Read columns directly: iterating rows would decode every page image just to get the id and text.
page_ids = ds_corpus["test"]["corpus_id"]
page_texts = [text or "" for text in ds_corpus["test"]["markdown"]]
print(f"{len(page_texts)} pages, longest {max(map(len, page_texts))} characters")


def to_pages(page_scores: np.ndarray, k: int) -> list[int]:
    """Ranked list of the top-k page corpus_ids for one score per page."""
    return [page_ids[i] for i in np.argsort(-page_scores)[:k]]

## Text retrievers (same as `rag.ipynb`)

BM25 keyword search and Qwen3-Embedding dense search over each page's `markdown`.

In [ ]:
import bm25s
import Stemmer

stemmer = Stemmer.Stemmer("english")

bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize(page_texts, stopwords="en", stemmer=stemmer))

def sparse_pages(query: str, k: int) -> list[int]:
    # return_ids=False gives plain string tokens, which get_scores maps through the index vocabulary
    q_tokens = bm25s.tokenize(query, stopwords="en", stemmer=stemmer, return_ids=False, show_progress=False)[0]
    return to_pages(bm25.get_scores(q_tokens), k)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
# Qwen3-Embedding is instruction-aware: queries get a task prompt, documents are embedded as-is.
QUERY_PROMPT = "Instruct: Given a question about a technical manual, retrieve the page that answers it\nQuery:"

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL, device=device, tokenizer_kwargs={"padding_side": "left"})
embedder.max_seq_length = 2048  # whole pages; longer pages are truncated
if device == "cuda":
    embedder.half()

page_emb = embedder.encode(page_texts, batch_size=8, normalize_embeddings=True, show_progress_bar=True)


def dense_pages(query: str, k: int) -> list[int]:
    q_emb = embedder.encode(query, prompt=QUERY_PROMPT, normalize_embeddings=True)
    # Dot product of the query vector with every page vector. Both are normalised to length 1,
    # so this equals cosine similarity: one score per page, higher means more similar.
    scores = np.dot(page_emb, q_emb)
    return to_pages(scores, k)

## Hybrid retrieval (page text + image captions)

The OCR `markdown` has no description of figures, diagrams or photos, only "Figure N" labels. A small vision-language model (`SmolVLM-500M-Instruct`) captions each page image, asked to describe the figures and tables rather than transcribe text. The captions are then searched with the same two methods as the text (BM25 and Qwen embeddings), and all four rankings (text BM25, text dense, caption BM25, caption dense) are merged with Reciprocal Rank Fusion: each page scores `sum 1 / (60 + rank)` over the rankings that return it. Captioning every page is the slow step (the dataset has no separate figure crops, so whole pages are captioned), so captions are cached in `page_captions.json` and the loop resumes where it stopped.

In [ ]:
import json
from pathlib import Path

from tqdm.auto import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor

CAPTION_MODEL = "HuggingFaceTB/SmolVLM-500M-Instruct"
CAPTION_PROMPT = (
    "Describe any figures, diagrams, charts, tables or photos on this page in 2-3 sentences, naming the objects, "
    "parts and labels shown. Do not transcribe the body text. If there are none, answer 'No figures.'"
)
CAPTION_BATCH = 8        # pages per generate() call; lower it if the GPU runs out of memory
CAPTION_MAX_TOKENS = 96
CAPTION_CACHE = Path("page_captions.json")  # corpus_id -> caption; makes this slow step resumable

captions = json.loads(CAPTION_CACHE.read_text()) if CAPTION_CACHE.exists() else {}
todo = [i for i, pid in enumerate(page_ids) if str(pid) not in captions]
print(f"{len(captions)} captions cached, {len(todo)} pages to caption")

if todo:
    cap_processor = AutoProcessor.from_pretrained(CAPTION_MODEL, do_image_splitting=False)  # one tile per page: faster
    cap_processor.tokenizer.padding_side = "left"  # batched generation needs left padding
    cap_model = AutoModelForImageTextToText.from_pretrained(
        CAPTION_MODEL, torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32
    ).to(device).eval()
    chat = cap_processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": CAPTION_PROMPT}]}],
        add_generation_prompt=True,
    )
    with torch.inference_mode():
        for n, start in enumerate(tqdm(range(0, len(todo), CAPTION_BATCH))):
            idx = todo[start : start + CAPTION_BATCH]
            images = [[img.convert("RGB")] for img in ds_corpus["test"][idx]["image"]]  # one image per prompt
            inputs = cap_processor(text=[chat] * len(idx), images=images, return_tensors="pt", padding=True).to(device)
            out = cap_model.generate(**inputs, max_new_tokens=CAPTION_MAX_TOKENS, do_sample=False)
            texts = cap_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for i, text in zip(idx, texts):
                captions[str(page_ids[i])] = text.strip()
            if n % 20 == 0:
                CAPTION_CACHE.write_text(json.dumps(captions))
    CAPTION_CACHE.write_text(json.dumps(captions))
    del cap_model, cap_processor
    torch.cuda.empty_cache()

# Pages without figures carry no signal, so drop their "No figures." answer instead of indexing it.
page_captions = ["" if c.lower().startswith("no figures") else c for c in (captions[str(pid)] for pid in page_ids)]
print(f"{sum(bool(c) for c in page_captions)} of {len(page_captions)} pages have a figure caption")
print(next(c for c in page_captions if c))

In [ ]:
# Caption indexes: same two retrievers as for the text, run over the (short) captions instead.
caption_bm25 = bm25s.BM25()
caption_bm25.index(bm25s.tokenize(page_captions, stopwords="en", stemmer=stemmer))
caption_emb = embedder.encode(page_captions, batch_size=32, normalize_embeddings=True, show_progress_bar=True)


def caption_sparse_pages(query: str, k: int) -> list[int]:
    q_tokens = bm25s.tokenize(query, stopwords="en", stemmer=stemmer, return_ids=False, show_progress=False)[0]
    return to_pages(caption_bm25.get_scores(q_tokens), k)


def caption_dense_pages(query: str, k: int) -> list[int]:
    q_emb = embedder.encode(query, prompt=QUERY_PROMPT, normalize_embeddings=True)
    return to_pages(np.dot(caption_emb, q_emb), k)


RRF_K = 60       # standard Reciprocal Rank Fusion constant
RRF_DEPTH = 100  # how many pages each retriever contributes to the fusion


def hybrid_pages(query: str, k: int) -> list[int]:
    """RRF over the page-text retrievers and the caption retrievers: score(page) = sum 1 / (RRF_K + rank)."""
    fused = defaultdict(float)
    for retrieve in (sparse_pages, dense_pages, caption_sparse_pages, caption_dense_pages):
        for rank, pid in enumerate(retrieve(query, RRF_DEPTH), 1):
            fused[pid] += 1 / (RRF_K + rank)
    return sorted(fused, key=fused.get, reverse=True)[:k]

## Benchmark

Page-level retrieval metrics against `qrels` (binary relevance), using the top 5 pages (`K = 5`):
- **recall@5:** share of a query's relevant pages found in the top 5.
- **nDCG@5:** how well the top 5 is ordered, with more credit for relevant pages ranked higher (1.0 is a perfect ordering).

In [ ]:
import math

import pandas as pd

K = 5
RETRIEVERS = {
    "sparse (BM25)": sparse_pages,
    "dense (qwen3-emb)": dense_pages,
    "hybrid (text + captions, RRF)": hybrid_pages,
}


def evaluate(retrieve_fn, k: int = K) -> dict:
    """Average recall@k and nDCG@k over all queries that have qrels."""
    recall = ndcg = 0.0
    n = 0
    for q in ds_queries["test"]:
        rel = qrels_by_query.get(q["query_id"])
        if not rel:
            continue
        ranked = retrieve_fn(q["query"], k)
        recall += len(rel & set(ranked)) / len(rel)
        dcg = sum(1 / math.log2(r + 1) for r, pid in enumerate(ranked, 1) if pid in rel)
        idcg = sum(1 / math.log2(r + 1) for r in range(1, min(len(rel), k) + 1))
        ndcg += dcg / idcg
        n += 1
    return {f"recall@{k}": recall / n, f"nDCG@{k}": ndcg / n}


results = pd.DataFrame({name: evaluate(fn) for name, fn in RETRIEVERS.items()}).T
results.round(3)